# 05 — Market-State Concentration Validation

Methodology gate before concentration conditioning.

Primary concentration is measured from a broad US common-equity market-state universe: CIZ common stocks on N/A/Q with positive market capitalisation, aggregated to PERMCO. The momentum portfolio keeps the stricter NYSE-size and $5 formation screens.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path('..').resolve() if Path.cwd().name == 'notebooks' else Path('.').resolve()
sys.path.append(str(ROOT))

from src.clean_monthly import CleaningConfig, build_clean_monthly_panel, build_market_state_panel
from src.concentration import calculate_monthly_concentration, largest_firms_by_month


## 1. Build the two distinct universes


In [ ]:
RAW = ROOT / 'data/raw/monthly_stock_common_equity_2000_2025_prod_v1.csv.gz'

market_state = build_market_state_panel(RAW)
investable, _ = build_clean_monthly_panel(RAW, CleaningConfig())

print(f'Market-state rows: {len(market_state):,}')
print(f'Investable rows:   {len(investable):,}')
print(f'Market-state PERMNO: {market_state.PERMNO.nunique():,}')
print(f'Investable PERMNO:   {investable.PERMNO.nunique():,}')


## 2. Compute company-level concentration on each universe


In [ ]:
conc_market = calculate_monthly_concentration(market_state)
conc_investable = calculate_monthly_concentration(investable)

compare = conc_market[['MthCalDt','N_Firms','N_Securities','Top10Share','HHI']].merge(
    conc_investable[['MthCalDt','N_Firms','N_Securities','Top10Share','HHI']],
    on='MthCalDt', suffixes=('_Market','_Investable'), validate='one_to_one'
)
compare['Top10_Diff'] = compare['Top10Share_Market'] - compare['Top10Share_Investable']
compare['HHI_Diff'] = compare['HHI_Market'] - compare['HHI_Investable']
compare.tail()


## 3. Selected year-end comparison


In [ ]:
years = [2000, 2008, 2020, 2025]
year_end = (
    compare[compare['MthCalDt'].dt.year.isin(years)]
    .sort_values('MthCalDt')
    .groupby(compare['MthCalDt'].dt.year)
    .tail(1)
)
year_end[['MthCalDt','N_Firms_Market','N_Firms_Investable','Top10Share_Market','Top10Share_Investable','Top10_Diff','HHI_Market','HHI_Investable','HHI_Diff']]


## 4. Time-series comparison


In [ ]:
fig, ax = plt.subplots(figsize=(10,4))
ax.plot(compare['MthCalDt'], compare['Top10Share_Market'], label='Broad market-state universe')
ax.plot(compare['MthCalDt'], compare['Top10Share_Investable'], label='Investable momentum universe', alpha=0.75)
ax.set_title('Top-10 company market-cap share: universe comparison')
ax.set_xlabel('Month')
ax.set_ylabel('Top10Share')
ax.legend()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(10,4))
ax.plot(compare['MthCalDt'], compare['HHI_Market'], label='Broad market-state universe')
ax.plot(compare['MthCalDt'], compare['HHI_Investable'], label='Investable momentum universe', alpha=0.75)
ax.set_title('Company-level HHI: universe comparison')
ax.set_xlabel('Month')
ax.set_ylabel('HHI')
ax.legend()
plt.show()


## 5. Broad-market top-company sanity check


In [ ]:
target_dates = pd.to_datetime(['2000-01-31','2010-01-29','2020-01-31','2025-12-31'])
largest_firms_by_month(market_state, n=10, months=target_dates)


## Methodology gate

If the broad-market series is economically sensible and free of obvious data artefacts, it becomes the primary concentration state series. The investable-universe series is retained only as a robustness comparison.
